# Ticha HTR — TrOCR fine-tuning

This notebook fine-tunes `microsoft/trocr-base-handwritten` on `maclenn77/ticha-htr-lines`, using its existing `train`, `validation`, and `test` splits.

It runs in **Google Colab** or on the **GCP VM** (`ticha-l4`, NVIDIA L4):

- **Colab:** choose **Runtime → Change runtime type → T4 GPU** (or another available GPU). Outputs go to Google Drive.
- **VM:** open it through the SSH tunnel and pick the **Python (ticha-trocr)** kernel (see `ENVIRONMENT.md`). Outputs go to `$TICHA_RUNS_DIR` (default `~/ticha-runs/`), where the other notebooks read them.

It trains three models, each from the pretrained checkpoint:

1. **First run**: 5 epochs, linear schedule.
2. **`smooth-cosine`**: label smoothing 0.1, cosine schedule, up to 12 epochs.
3. **`smooth-cosine-aug`**: the same recipe on the original plus the augmented train lines. This is the model used in the report, decoded with wider beams and a right-to-left reranker in [`ticha_trocr_beams_r2l_rerank.ipynb`](ticha_trocr_beams_r2l_rerank.ipynb).

Settings, data cleaning and model pieces shared with the other training notebooks are in this folder: `trocr_setup.py` (paths, model, dataset version, seed), `trocr_data.py` (loading, cleaning and checks) and `trocr_model.py` (collators, model setup, metrics). The training hyperparameters stay in this notebook.

The final comparison of all models on the test set is in [`../reports/ticha_trocr_evaluation.ipynb`](../reports/ticha_trocr_evaluation.ipynb).

## Setup

In Colab, clones the repository (for the shared modules and `src/`) and installs the packages. On the VM the kernel already has them.

In [ ]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # Colab starts without the repository: clone it for the shared modules and src/
    if not Path('/content/ticha').exists():
        !git clone -q --depth 1 https://github.com/Maclenn77/ticha.git /content/ticha
    %pip -q install "transformers==4.46.3" "datasets>=2.19,<4" "accelerate>=0.26" "jiwer>=3.0"
    ROOT = Path('/content/ticha')
else:
    # The kernel may start in the repository root or in notebooks/train
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'notebooks' / 'train' / 'trocr_setup.py').exists())
sys.path[:0] = [str(ROOT / 'notebooks' / 'train'), str(ROOT / 'src')]

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

import gc
import os

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from transformers import EarlyStoppingCallback, Seq2SeqTrainer, Seq2SeqTrainingArguments, TrOCRProcessor

from trocr_data import (check_document_overlap, drop_train_docs, duplicate_train_docs, load_clean_splits,
                        load_train_aug, translation_pair_docs)
from trocr_model import (TrOCRDataCollator, TrOCRSmoothingCollator, cer_wer_metrics, max_target_length,
                         pretrained_model)
from trocr_setup import MODEL_ID, RUNS_DIR, SEED, gpu_memory_gb, require_gpu

print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())

## Data: load, clean and check

`load_clean_splits` loads the `train`, `validation` and `test` splits of the dataset at `DATASET_REVISION` and cleans the transcriber notes with `ticha.core.htr.clean_text`. The transcriptions mark expansions, glosses and damage with `[]` and `()`: expansions and glosses are deleted, so the text matches what is written on the page (diplomatic text), and lines with damage, illegible text or editorial notes are dropped, because their image has ink the text doesn't cover. Uncertain readings such as `(Santigo?)` are kept as text in train and dropped from validation and test, so the reported CER is on confident transcriptions only. The Hub dataset stays as published; the original text is kept in `text_raw`.

`check_document_overlap` stops if a document is in two splits: its hand would leak into evaluation.

In [ ]:
ds = load_clean_splits()
check_document_overlap(ds)

In [ ]:
# One sample, some cleaned lines and the transcription lengths
sample = ds['train'][0]
print('Sample text:', repr(sample['text']))
print('Document:', sample['doc_id'], '| language:', sample.get('language'), '| source:', sample.get('source'))
display(sample['image'].convert('RGB'))

for raw, text in [(r, t) for r, t in zip(ds['train']['text_raw'], ds['train']['text']) if r != t][:5]:
    print(f'{raw!r}\n  -> {text!r}')

for split_name, split_data in ds.items():
    lengths = np.array([len(t) for t in split_data['text']])
    print(f"{split_name}: p50={np.percentile(lengths, 50):.0f} chars, p95={np.percentile(lengths, 95):.0f} chars, "
          f"max={lengths.max()}")

## Remove train documents that duplicate evaluation documents

Ticha catalogues some manuscripts twice. A train document is dropped when more than half of its lines appear verbatim in validation or test.

Original/translation pairs (`Al710` and `Al710T`) are different manuscripts in different languages, but they share names, places and dates. They are kept by default (about 10% of the train lines); set `DROP_TRANSLATION_PAIRS = True` to drop the train half of each pair and report both numbers if the difference matters.

In [ ]:
DUPLICATE_SHARE = 0.5           # share of a train document's lines found in validation or test
DROP_TRANSLATION_PAIRS = False  # also drop train documents whose translation is in validation or test

duplicate_docs = duplicate_train_docs(ds, share=DUPLICATE_SHARE)
pair_docs = translation_pair_docs(ds)
print('Train documents duplicating validation/test documents (dropped):', sorted(duplicate_docs))
print(f"Train documents paired with a validation/test translation: {len(pair_docs)}, "
      f"{'dropped' if DROP_TRANSLATION_PAIRS else 'kept'}")
ds = drop_train_docs(ds, duplicate_docs | (pair_docs if DROP_TRANSLATION_PAIRS else set()))

## Training

### Load the pretrained TrOCR model

The base handwritten checkpoint is a transfer-learning starting point, not a Zapotec-specific model. The label length comes from the train texts: the 99th percentile of their token counts plus 8, at least 64 (`max_target_length` in `trocr_model.py`).

In [ ]:
processor = TrOCRProcessor.from_pretrained(MODEL_ID)
MAX_TARGET_LENGTH, p99_tokens, truncated_count = max_target_length(processor.tokenizer, ds['train']['text'])
print('99th percentile token length:', p99_tokens)
print('MAX_TARGET_LENGTH:', MAX_TARGET_LENGTH)
print('Training lines longer than this limit:', truncated_count, '/', len(ds['train']))

# Special tokens, and the decoding that scores validation during training: 2 beams
model = pretrained_model(MODEL_ID, processor.tokenizer, MAX_TARGET_LENGTH, num_beams=2)
# Builds each batch from the images and texts when it is needed (no image tensors held in RAM)
data_collator = TrOCRDataCollator(processor, MAX_TARGET_LENGTH)
compute_metrics = cer_wer_metrics(processor)  # validation CER and WER
print('Loaded:', MODEL_ID)

In [ ]:
# Guard against accidental CPU training.
require_gpu('in Colab choose Runtime > Change runtime type > T4 GPU; on the VM check nvidia-smi. Then rerun the cells')

if IN_COLAB:
    OUTPUT_DIR = '/content/drive/MyDrive/TichaHTR/trocr-base-handwritten-finetuned'
else:
    # Every notebook reads the runs from RUNS_DIR (TICHA_RUNS_DIR); TICHA_OUTPUT_DIR overrides this run's folder only.
    OUTPUT_DIR = os.environ.get('TICHA_OUTPUT_DIR') or str(RUNS_DIR / 'trocr-base-handwritten-finetuned')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Same effective batch size (8) everywhere: 2 x 4 on a 16 GB T4, 8 x 1 on a 23 GB L4.
BATCH_SIZE = 8 if gpu_memory_gb() >= 20 else 2
# bf16 is more stable than fp16 and the L4 (Ada) supports it; the T4 does not.
USE_BF16 = torch.cuda.is_bf16_supported()
print(f'GPU memory: {gpu_memory_gb():.0f} GB | batch size: {BATCH_SIZE} | bf16: {USE_BF16}')

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=8 // BATCH_SIZE,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.05,
    num_train_epochs=5,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=2,
    fp16=not USE_BF16,
    bf16=USE_BF16,
    remove_unused_columns=False,
    dataloader_num_workers=2 if IN_COLAB else 4,
    report_to='none',
    seed=SEED
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=ds['train'],
    eval_dataset=ds['validation'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)
print('Training output:', OUTPUT_DIR)

### Optional: measure the pretrained baseline before fine-tuning

Run this cell **before** the training cell to get the zero-shot score of the pretrained checkpoint on the test split. Don't use the test score to tune hyperparameters.

In [ ]:
# Optional; can take several minutes because predictions are generated for the full test set.
baseline_result = trainer.predict(ds['test'], metric_key_prefix='baseline')
print(baseline_result.metrics)
trainer.save_metrics('baseline', baseline_result.metrics)


### First run

In [ ]:
# Start fine-tuning. Checkpoints and logs are saved to OUTPUT_DIR (Google Drive in Colab).
train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
trainer.log_metrics('train', train_result.metrics)
trainer.save_metrics('train', train_result.metrics)
trainer.save_state()
print('Saved model and processor to:', OUTPUT_DIR)

## Experiment: label smoothing, cosine schedule and augmentation

In [ ]:
# Experiment: same model, modified training parameters, without and with the augmented lines.
# Run after the fine-tuning cell above; it leaves `trainer` (the first fine-tuned model) untouched,
# so the test cell below still evaluates that model.
# Changes against the first run (one recipe, two datasets, so the only difference is augmentation):
#   - label smoothing 0.1: the first run overfit (train loss 0.30, val loss flat at 1.42 from epoch 4)
#   - cosine schedule instead of linear: less time at a near-zero learning rate
#   - up to 12 epochs; early stopping on validation CER (patience 2) keeps the best checkpoint
#   - AdamW and lr 5e-5 as before (AdamW is the Trainer default; written out here)
# Each run starts from the pretrained checkpoint, not from the model fine-tuned above.
# With label smoothing the Trainer removes `labels` before calling the model, so TrOCRSmoothingCollator also
# passes the decoder inputs.
# The original train lines plus their augmented copies (train documents only), cleaned like the originals
# and shuffled with SEED; each epoch takes as many times longer as the set is larger.
train_aug = load_train_aug(ds['train'])

EXPERIMENTS = {
    'smooth-cosine': ds['train'],
    'smooth-cosine-aug': train_aug,
}
experiment_results = {}

for name, train_set in EXPERIMENTS.items():
    run_dir = f'{OUTPUT_DIR}-{name}'
    os.makedirs(run_dir, exist_ok=True)
    exp_args = Seq2SeqTrainingArguments(
        output_dir=run_dir,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=8 // BATCH_SIZE,
        optim='adamw_torch',
        learning_rate=5e-5,
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        warmup_ratio=0.05,
        label_smoothing_factor=0.1,
        num_train_epochs=12,
        eval_strategy='epoch',
        save_strategy='epoch',
        logging_strategy='steps',
        logging_steps=100,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model='cer',
        greater_is_better=False,
        predict_with_generate=True,
        generation_max_length=MAX_TARGET_LENGTH,
        generation_num_beams=2,
        fp16=not USE_BF16,
        bf16=USE_BF16,
        remove_unused_columns=False,
        dataloader_num_workers=2 if IN_COLAB else 4,
        report_to='none',
        seed=SEED
    )
    exp_trainer = Seq2SeqTrainer(
        model=pretrained_model(MODEL_ID, processor.tokenizer, MAX_TARGET_LENGTH, num_beams=2),
        args=exp_args,
        train_dataset=train_set,
        eval_dataset=ds['validation'],
        data_collator=TrOCRSmoothingCollator(processor, MAX_TARGET_LENGTH),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )
    print(f'\n=== {name}: {len(train_set):,} train lines -> {run_dir}')
    exp_trainer.train()
    exp_trainer.save_model(run_dir)  # the best checkpoint (load_best_model_at_end)
    processor.save_pretrained(run_dir)
    exp_trainer.save_state()

    evals = [h for h in exp_trainer.state.log_history if 'eval_cer' in h]
    best = min(evals, key=lambda h: h['eval_cer'])
    experiment_results[name] = {
        'best_epoch': int(round(best['epoch'])), 'epochs_run': int(round(evals[-1]['epoch'])),
        'val_loss': best['eval_loss'], 'val_cer': best['eval_cer'], 'val_wer': best['eval_wer'],
        'model_dir': run_dir,
    }
    del exp_trainer
    gc.collect()
    torch.cuda.empty_cache()

# Compare with the first run. Its val loss is not comparable to the others: label smoothing raises the loss.
first = [h for h in trainer.state.log_history if 'eval_cer' in h]
first_best = min(first, key=lambda h: h['eval_cer'])
experiment_results = {
    'first run (linear, no smoothing)': {
        'best_epoch': int(round(first_best['epoch'])), 'epochs_run': int(round(first[-1]['epoch'])),
        'val_loss': first_best['eval_loss'], 'val_cer': first_best['eval_cer'],
        'val_wer': first_best['eval_wer'], 'model_dir': OUTPUT_DIR,
    },
    **experiment_results,
}
display(pd.DataFrame(experiment_results).T)


## Test evaluation of the first run

This scores the model of the first run. All the models, including the two experiments, are compared in [`../reports/ticha_trocr_evaluation.ipynb`](../reports/ticha_trocr_evaluation.ipynb).

In [ ]:
# Final test evaluation — run after training has completed.
test_result = trainer.predict(ds['test'], metric_key_prefix='test')
print(test_result.metrics)

pred_ids = test_result.predictions
if isinstance(pred_ids, tuple):
    pred_ids = pred_ids[0]
predictions = processor.batch_decode(pred_ids, skip_special_tokens=True)
references = [str(t or '').strip() for t in ds['test']['text']]

results_df = pd.DataFrame({
    'id': ds['test']['id'],
    'doc_id': ds['test']['doc_id'],
    'reference': references,
    'prediction': [p.strip() for p in predictions],
    'language': ds['test']['language']
})
predictions_path = os.path.join(OUTPUT_DIR, 'test_predictions.csv')
results_df.to_csv(predictions_path, index=False)
print('Saved predictions:', predictions_path)
display(results_df.head(10))


## How to report results

- CER and WER are error rates: **lower is better**.